# Kisherceg Chatbot, Sentence Embedding, Koszinusz-hasonlóság, Átfedéssel módosított chunkolás


In [1]:
# Könyvtárak telepítése
!pip install sentence-transformers scikit-learn --quiet

In [2]:
# Importok
from sentence_transformers import SentenceTransformer
import numpy as np
import requests
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# Kisherceg letöltése
url = "https://raw.githubusercontent.com/minorharpman/ai_prog_pub/main/hirlevel_12/kisherceg.txt"
# Robinson letöltése
#url = "https://raw.githubusercontent.com/minorharpman/ai_prog_pub/main/hirlevel_12/robinson.txt"
text = requests.get(url).text

print(text[92:300])



Kérem a gyerekeket, ne haragudjanak, amiért ezt a könyvet egy fölnõttnek ajánlom. Komoly mentségem van rá: ez a fölnõtt széles e világon a legjobb barátom. De van egy másik mentségem is: ez a fölnõtt mind


## Szöveg chunkolása

In [4]:
# Mondatokra bontás
sentences = text.split(".")
# csak a 20 karakternál hosszabb mondatokat tartom meg.
sentences = [s.strip() for s in sentences if len(s.strip()) > 20]


# Chunk méret (hány mondat egy blokkban)
chunk_size = 5
chunks = []

for i in range(0, len(sentences), chunk_size):
    # alap chunk (5 mondat)
    core = sentences[i:i+chunk_size]

    # előző chunk utolsó mondata (ha van)
    prev_overlap = [sentences[i-1]] if i > 0 else []

    # következő chunk első mondata (ha van)
    next_overlap = [sentences[i+chunk_size]] if i + chunk_size < len(sentences) else []

    # összeállítás
    chunk_sentences = prev_overlap + core + next_overlap

    chunk = ". ".join(chunk_sentences)
    chunks.append(chunk)

print(f"Chunkok száma: {len(chunks)}")
print(chunks[1])


Chunkok száma: 230
Nagy szüksége van vigasztalásra. Ha pedig ez a sok mentség nem elegendõ, akkor annak a gyereknek ajánlom könyvemet, aki valaha ez a fölnõtt volt. Mert elõbb minden fölnõtt gyerek volt. (De csak kevesen emlékeznek rá. ) Ajánlásomat tehát kijavítom, ilyesformán:

Léon Werth-nek,
amikor még kisfiú volt. Hatéves koromban egy könyvben, mely az õserdõrõl szólt, és Igaz Történetek volt a címe, láttam egy nagyszerû képet. Óriáskígyót ábrázolt, amint egy vadállatot nyel el


## Embeddingek készítése

In [5]:
# főleg angol adatokon lett tanítva, magyarnál egyszerű hasonlóság keresést  kezeli
#model = SentenceTransformer('all-MiniLM-L6-v2')
# több nyelvet (50 nyelv, európai nyelvekben jó teljesítmény ) támogató modell
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Chunk embeddingek
chunk_embeddings = model.encode(chunks, show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

In [6]:
# a "magány" megkeresése az embeddingben
index = next(i for i, doc in enumerate(chunks) if "magány" in doc)
index
my_embedding = chunk_embeddings[index]
#my_embedding
# 384 dimenziós vektor
print("Embedding shape, :", my_embedding.shape)

Embedding shape, : (384,)


## Kereső függvény (Koszinusz-hasonlóság)


In [7]:
def search(query, top_k=3):
    query_emb = model.encode([query])

    # koszinusz hasonlóság számítás
    similarities = cosine_similarity(query_emb, chunk_embeddings)[0]

    top_indices = np.argsort(similarities)[-top_k:][::-1]

    results = []
    for idx in top_indices:
        results.append((similarities[idx], chunks[idx]))

    return results

## Tesztelés

In [13]:
# példa: magány, hiúság
# angol keresésre is működik paraphrase-multilingual-MiniLM-L12-v2 modell használata esetén. pl: loneliness, vanity
query = input("Írj be egy szöveget: ")

results = search(query)

for score, text in results:
    print("\n---")
    print(f"Hasonlóság: {score:.4f}")
    print(text)

Írj be egy szöveget: csillagász

---
Hasonlóság: 0.5775
- Légy?

- Dehogy! Olyan kis csillogó. - Méhek?

- Dehogy! Azok az aranyos kis izék, amin a semmittevõk ábrándozni szoktak. Én azonban komoly ember vagyok! Nekem nincs idõm semmiféle ábrándozásra. - És mit csinálsz azzal az ötszázmillió csillaggal?

- Ötszázegymillió-hatszázhuszonkétezer-hétszázharmincegy. Komoly ember vagyok, szeretem a pontosságot. - Mit csinálsz ezekkel a csillagokkal?

- Hogy mit csinálok velük?

- Igen. - Birtoklod a csillagokat?

- Igen

---
Hasonlóság: 0.5751
- Mit akarsz ezzel mondani?

- Az embereknek nem ugyanazt jelentik a csillagaik. Akik úton járnak, azoknak vezetõül szolgálnak a csillagok. Másoknak nem egyebek csöppnyi kis fényeknél. Ismét mások, a tudósok számára problémák. Az üzletemberem szemében aranyból voltak. A csillagok viszont mind-mind hallgatnak. De neked olyan csillagaid lesznek, amilyenek senki másnak

---
Hasonlóság: 0.5436
Például elkereszteli "a 3251. Minden okom megvan rá, hogy azt h